In [2]:
import pandas as pd

df = pd.read_csv('MetObjects_with_country.csv')

C:\Users\craig\AppData\Local\Temp\ipykernel_17248\11072977.py:3: DtypeWarning: Columns (0: Gallery Number, 1: AccessionYear, 2: Culture, 3: Period, 4: Dynasty, 5: Reign, 6: Constituent ID, 7: City, 8: County, 9: Subregion, 10: Locale, 11: Locus, 12: Excavation, 13: River, 14: Classification, 15: Rights and Reproduction) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('MetObjects_with_country.csv')


In [3]:
replacements = {
    r'^Ira$':                 'Iran',
    r'^Chi$':                 'China',
    r'^Japa$':                'Japan',
    r'^Cameroo$':             'Cameroon',
    r'^Gha$':                 'Ghana',
    r'^Uzbekista$':           'Uzbekistan',
    r'^Republic of Beni$':    'Republic of Benin',
    r'^Gabo$':                'Gabon',
    r'^Yeme$':                'Yemen',
    r'^Argenti$':             'Argentina',
    r'^Republic of Cameroo$': 'Cameroon',
    r'^Suda$':                'Sudan',
    r'^Taiwa$':               'Taiwan',
    r'^Swede$':               'Sweden',
    r'^United States$':       'United States of America',
    r'^England$':             'United Kingdom',
    r'^Spai$':                'Spain',
    r'^Czech Republic$':      'Czechia',
    r'^Democratic Republic of the Congo$' : 'Congo',
    r'^Afghanista$' :         'Afghanistan',
    r'^Alamania$':            'Germany',
    r'^Arabia$':              'Saudi Arabia',
    r'^Azerbaija$':           'Azerbaijan',
    r'^Bhuta$':               'Bhutan',
    r'^Beni$':                'Benin',
    r'^Bohemia$':             'Czechia',
    r'^Borneo$':              'Indonesia',
    r'^Byzantine Egypt$':     'Egypt',
    
    
}

df['country_clean'] = df['country_clean'].replace(replacements, regex=True)


In [4]:
print(df['country_clean'].value_counts().to_string())

country_clean
Egypt                                           33416
United States of America                         9944
Iran                                             6807
Peru                                             3429
France                                           2173
Mexico                                           1572
India                                            1566
United Kingdom                                   1454
Indonesia                                        1442
Turkey                                            959
China                                             950
Germany                                           940
Papua New Guinea                                  883
Nigeria                                           656
Italy                                             585
Syria                                             544
Congo                                             534
Iraq                                              462
Spain         

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 76007 entries, 0 to 76006
Data columns (total 56 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Object Number            76007 non-null  str    
 1   Is Highlight             76007 non-null  bool   
 2   Is Timeline Work         76007 non-null  bool   
 3   Is Public Domain         76007 non-null  bool   
 4   Object ID                76007 non-null  int64  
 5   Gallery Number           30825 non-null  object 
 6   Department               76007 non-null  str    
 7   AccessionYear            74802 non-null  object 
 8   Object Name              75687 non-null  str    
 9   Title                    76006 non-null  str    
 10  Culture                  31914 non-null  str    
 11  Period                   28146 non-null  str    
 12  Dynasty                  23185 non-null  str    
 13  Reign                    11231 non-null  str    
 14  Portfolio                0 non-nu

In [6]:
df['Credit Line'].value_counts()

Credit Line
The Crosby Brown Collection of Musical Instruments, 1889                                  2857
Gift of Helen Miller Gould, 1910                                                          2554
Rogers Fund, 1948                                                                         2529
The Michael C. Rockefeller Memorial Collection, Bequest of Nelson A. Rockefeller, 1979    1665
Rogers Fund, 1917                                                                         1616
                                                                                          ... 
Gift of Brian and Florence Mahony, 2022                                                      1
Gift of Leslie M. Beebe and Bruce Nussbaum, 2022                                             1
Watson Library copy: Gift of Mary C. Schlosser                                               1
Watson Library copy: Arthur K. Watson Gift                                                   1
Purchase, Ravi and Seran Trehan Gift, 

In [7]:
df['Credit Year'] = df['Credit Line'].str.extract(r'(\d{4})')
df['Credit Year'] = pd.to_numeric(df['Credit Year'])

In [8]:
df['Donor_Raw'] = df['Credit Line'].str.replace(
    r',?\s*\d{4}.*', 
    '', 
    regex=True
).str.strip()

In [9]:
df['Donor_Raw'].value_counts().head(20)


Donor_Raw
Rogers Fund                                                                                                   21903
Rogers Fund and Edward S. Harkness Gift                                                                        3183
The Crosby Brown Collection of Musical Instruments                                                             2860
Gift of Helen Miller Gould                                                                                     2554
The Michael C. Rockefeller Memorial Collection, Bequest of Nelson A. Rockefeller                               1666
Purchase, Edward S. Harkness Gift                                                                              1459
Gift of J. Pierpont Morgan                                                                                     1405
Theodore M. Davis Collection, Bequest of Theodore M. Davis                                                     1068
The Cloisters Collection                                      

In [10]:
df_export = df[['Title', 'country_clean']]
df_export.to_csv('MetObjects_slim.csv', index=False)

In [11]:
df.to_csv('MET_country_cleaned.csv')

In [12]:
df['country_clean'].value_counts().reset_index().rename(columns={'index': 'country', 'country_clean': 'count'}).to_csv('country_counts.csv', index=False)


In [18]:
# Creating a Pivot of Donor/gift/fund by country 

top_donors = df['Donor_Raw'].value_counts().head(20)
pivot = (df[df['Donor_Raw'].isin(top_donors)]
         .groupby(['Donor_Raw', 'country_clean'])
         .size()
         .unstack(fill_value=0))



In [20]:
fund = 'Rogers Fund'
df[df['Donor_Raw'] == fund]['country_clean'].value_counts().head(20)


country_clean
Egypt                       14866
Iran                         4680
United States of America      658
Iraq                          331
Syria                         210
India                         160
United Kingdom                115
Turkey                        115
Peru                           93
Italy                          70
Spain                          65
Indonesia                      64
France                         48
Uzbekistan                     48
China                          42
Nubia                          36
Morocco                        35
Germany                        35
Papua New Guinea               32
Netherlands                    29
Name: count, dtype: int64

In [21]:
top_donors = df['Donor_Raw'].value_counts().head(10).index
breakdown = (df[df['Donor_Raw'].isin(top_donors)]
             .groupby(['Donor_Raw', 'country_clean'])
             .size()
             .reset_index(name='count')
             .sort_values(['Donor_Raw', 'count'], ascending=[True, False]))


In [22]:
breakdown.to_csv('donor_country_breakdown.csv', index=False)


In [24]:
# Keep only top 10 countries across the whole dataset
top_countries = df['country_clean'].value_counts().head(10).index

breakdown['country_grouped'] = breakdown['country_clean'].where(
    breakdown['country_clean'].isin(top_countries), 'Other'
)

pivot = (breakdown.groupby(['Donor_Raw', 'country_grouped'])['count']
         .sum()
         .unstack(fill_value=0))

pivot.to_csv('donor_country_pivot.csv')

